In [1]:
from dotenv import load_dotenv
import os

# 1. Load the environment variables from the .env file
load_dotenv()

# 2. Access the variables using os.environ
hf_token = os.environ.get("HF_TOKEN")
db_url = os.environ.get("DATABASE_URL")

In [2]:
from huggingface_hub import login
login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import h5py
import numpy as np

def load_h5(h5_file_path, verbose=False):
    features = []
    labels = []
    filenames = []

    with h5py.File(h5_file_path, "r") as hf:
        for i in hf.keys():
            features.append(hf[i][:])
            labels.append(hf[i].attrs['label'])
            filenames.append(i)

    features = np.array(features)
    labels = np.array(labels)
    filenames = np.array(filenames)

    try:
        assert len(features) == len(labels) == len(filenames), "Data, labels, and filenames must have the same length."
    except AssertionError as e:
        print(f"AssertionError: {e}")
        print(f"Length of data: {len(features)}")
        print(f"Length of labels: {len(labels)}")
        print(f"Length of filenames: {len(filenames)}")
        raise

    if verbose:
        print(f"Loaded {len(features)} samples from {h5_file_path}")
        print(f"\tFeatures shape: {features.shape}")
        print(f"\tLabels shape: {labels.shape}")
        print(f"\tFilenames shape: {filenames.shape}")

        # get the unique labels and their counts
        unique_labels, counts = np.unique(labels, return_counts=True)
        print("\tLabel distribution:")
        for label, count in zip(unique_labels, counts):
            print(f"\t\tLabel {label}: {count} samples")

    return features, labels, filenames

data, labels, filenames = data  = load_h5("./feats_batch1/virchow2.h5", True)

Loaded 7281 samples from ./feats_batch1/virchow2.h5
	Features shape: (7281, 2560)
	Labels shape: (7281,)
	Filenames shape: (7281,)
	Label distribution:
		Label 0: 2972 samples
		Label 1: 4309 samples


In [4]:
# MODELS=virchow2,ctranspath,hoptimus0,hoptimus1,uni_v2,musk,conch_v15,hibou-l

for model in ["virchow2", "ctranspath", "hoptimus0", "hoptimus1", "uni_v2", "musk", "conch_v15", "hibou-l"]:
    data, labels, filenames = load_h5(f"./feats_batch1/{model}.h5", True)
    break

Loaded 7281 samples from ./feats_batch1/virchow2.h5
	Features shape: (7281, 2560)
	Labels shape: (7281,)
	Filenames shape: (7281,)
	Label distribution:
		Label 0: 2972 samples
		Label 1: 4309 samples


In [5]:
for model in ["virchow2", "ctranspath", "hoptimus0", "hoptimus1", "uni_v2", "musk", "conch_v15", "hibou-l"]:
    data, labels, filenames = load_h5(f"./feats_batch2/{model}.h5", True)
    break

Loaded 52944 samples from ./feats_batch2/virchow2.h5
	Features shape: (52944, 2560)
	Labels shape: (52944,)
	Filenames shape: (52944,)
	Label distribution:
		Label 0: 36253 samples
		Label 1: 16691 samples


In [ ]:
import os
import sys
from PIL import Image, ImageFile, PngImagePlugin
Image.MAX_IMAGE_PIXELS = None 
PngImagePlugin.MAX_TEXT_CHUNK = 100 * 1024 * 1024  # 100MB
PngImagePlugin.MAX_TEXT_MEMORY = 100 * 1024 * 1024 # 100MB
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import random
from pathlib import Path    
from helpers import CustomDataset, custom_extract_patch_features_from_dataloader, get_model_and_transform

# ... rest of your get_feats.py imports and code (e.g., AutoModel.from_pretrained)

DATA_DIR = Path("/mnt/Z/cuhk_data/HPACG/batch1/data")


chosen_model = "virchow2"
print("Using model", chosen_model)
model, trnsfrms_val = get_model_and_transform(chosen_model)
model.to('cuda')
model.eval()

Using model virchow2


Virchow2InferenceEncoder(
  (model): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (mlp): GluMlp(
          (fc1): Linear(in_features=1280, out_features=6832, bias=True)
          (act): 

In [52]:
type(model.model.blocks)

torch.nn.modules.container.Sequential

In [50]:
type(model)

trident.patch_encoder_models.load.Virchow2InferenceEncoder

In [54]:
model.parameters

<bound method Module.parameters of Virchow2InferenceEncoder(
  (model): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): LayerScale()
        (drop_path1): Identity()
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (mlp): GluMlp(
          (fc1): Linear(in_features=1280, out_features

In [61]:
for _, i in model.model.blocks[-1].attn.named_parameters():
    print(i.requires_grad)

True
True
True
True


In [46]:
def get_random_img_path():
    rand_img = random.choice(os.listdir(DATA_DIR / "positive"))
    rand_img_full_path = DATA_DIR / "positive" / rand_img

    return rand_img_full_path

In [47]:
get_random_img_path()

PosixPath('/mnt/Z/cuhk_data/HPACG/batch1/data/positive/23S001042-III.svs_2048_1536_0_512_512_1.png')

In [31]:
os.listdir("/mnt/Z/cuhk_data/HPACG/batch1/data/positive")

['23S000216-II.svs_59904_45568_0_512_512_1.png',
 '23S000216-II.svs_59904_46080_0_512_512_1.png',
 '23S000216-II.svs_59904_46592_0_512_512_1.png',
 '23S000216-II.svs_59904_47104_0_512_512_1.png',
 '23S000216-II.svs_59904_47616_0_512_512_1.png',
 '23S000216-II.svs_59904_49152_0_512_512_1.png',
 '23S000216-II.svs_60416_45056_0_512_512_1.png',
 '23S000216-II.svs_60416_45568_0_512_512_1.png',
 '23S000216-II.svs_60416_47104_0_512_512_1.png',
 '23S000216-II.svs_60416_48640_0_512_512_1.png',
 '23S000216-II.svs_60416_49152_0_512_512_1.png',
 '23S000216-II.svs_60416_49664_0_512_512_1.png',
 '23S000216-II.svs_60928_44544_0_512_512_1.png',
 '23S000216-II.svs_60928_45056_0_512_512_1.png',
 '23S000216-II.svs_60928_49152_0_512_512_1.png',
 '23S000216-II.svs_60928_49664_0_512_512_1.png',
 '23S000216-II.svs_61440_44032_0_512_512_1.png',
 '23S000216-II.svs_61952_43520_0_512_512_1.png',
 '23S000216-II.svs_61952_50176_0_512_512_1.png',
 '23S000216-II.svs_61952_50688_0_512_512_1.png',
 '23S000216-II.svs_6

In [ ]:
from trident.patch_encoder_models import encoder_factory

In [ ]:
from helpers import CustomDataset, get_model_and_transform
import torch

In [ ]:
model, trns = get_model_and_transform("musk") 

dataDir = "/Z/cuhk_data/HPACG/final"
train_dataset = CustomDataset(dataDir, transform=trns)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=False, num_workers=8)

In [ ]:
model, trns = get_model_and_transform("virchow2")

In [ ]:
len(model.model.blocks)

In [ ]:
model.model.blocks[31].attn

In [ ]:
from draw_attn import draw_attention_mask

In [ ]:
model2, trns2 = get_model_and_transform("conch_v15")

In [ ]:
draw_attention_mask(model2, trns2, "/Z/cuhk_data/HPACG/final/train/positive/23S000216-II.svs_59904_45568_0_512_512_1.png", "conch_v15")